### RAG Pipelines - Data Ingestion to Vector DB Pipeline 

In [4]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader, TextLoader, CSVLoader
import langchain
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

 

In [8]:
### Read all pdf's inside the directory

def process_all_pdf(pdf_directory):
    """this function reads all pdf from the given directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all pdf files recursively
    pdf_files = list(pdf_dir.glob("**/*.csv"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing file {pdf_file.name}")

        try:
            loader = CSVLoader(str(pdf_file))
            documents = loader.load()

            # add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type']  = 'pdf'

            all_documents.extend(documents)
            print(f"Successfully Loaded {len(documents)} pages")
        
        except Exception as e:
            print(f"Error in loading {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all pdf in the data directory
all_pdf_documents = process_all_pdf("../data")

content = [doc.page_content for doc in all_pdf_documents]
print(content)

Found 1 PDF files to process

Processing file Sold Records.csv
Successfully Loaded 2 pages

Total documents loaded: 2
['ID: 1\nName: ICICI PRUDENTIAL SILVER ETF FUND OF FUND DIRECT PLAN GROWTH\nInvest amount: 1500\nSold amount: 2214.84\ndifference amount: 714.84\nInvest Start date: 01-Jul\nSold Date: 16-Oct-25\nHolding Period in Dates: 107', 'ID: 2\nName: ICICI PRUDENTIAL REGULAR GOLD SAVINGS FUND (FOF)\nInvest amount: 3500\nSold amount: 4588\ndifference amount: 1088\nInvest Start date: \nSold Date: 04-Nov-25\nHolding Period in Dates: ']


In [4]:
### Text Splitting into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split the documents into smaller chunks for RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [5]:
chunks = split_documents(all_pdf_documents)
chunks

Split 33 documents into 79 chunks

Example chunk:
Content: Money Market in India 
 
 
Prof. Shailendra Singh Bhadouria 
Professor & Ex Head, Department of Commerce 
Ex Dean, Faculty of Commerce & Management 
Indira Gandhi National Tribal University 
(A Centra...
Metadata: {'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2020-03-16T15:47:03+00:00', 'author': 'Shailendra Singh', 'moddate': '2020-03-16T15:47:05+00:00', 'source': '..\\data\\pdf\\Money Market.pdf', 'total_pages': 18, 'page': 0, 'page_label': '1', 'source_file': 'Money Market.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2020-03-16T15:47:03+00:00', 'author': 'Shailendra Singh', 'moddate': '2020-03-16T15:47:05+00:00', 'source': '..\\data\\pdf\\Money Market.pdf', 'total_pages': 18, 'page': 0, 'page_label': '1', 'source_file': 'Money Market.pdf', 'file_type': 'pdf'}, page_content='Money Market in India \n \n \nProf. Shailendra Singh Bhadouria \nProfessor & Ex Head, Department of Commerce \nEx Dean, Faculty of Commerce & Management \nIndira Gandhi National Tribal University \n(A Central University) \nAmarkantak (MP) 484887 \n \n \n \nMeaning of Money Market: \nMoney market is a market for short -term funds. We define the short -term as a period of 364 days or \nless. In other words, the borrowing and repayment take place in 364 days or less. The manufacturers \nneed two types of finance: finance to meet daily expenses like purchase of raw material, payment of \nwages, excise duty, electricity charges etc

### Embeddings and VectorStoreDB

In [1]:
import numpy as np
import chromadb
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

d:\Documents\Gen AI - RAG, Vector DB, Agentic AI\chapter 1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
class EmbeddingManager:
    """
    Handles document embedding using Sentence Transformer from Hugging Face
    """
    def __init__(self, model_name: str = "all-MiniLM-L6-v2") -> None:
        """
        Iniatialize the embedding manager

        input:
            model_name: Hugging face model for sentence embedding
        """
        self.model_name = model_name
        self.model = None
        self._load_model() # protected method
        
    def _load_model(self) -> None:
        """
        Load the sentence transformer model
        """

        try:
            print(f"Loading Embedded model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error during loading model {self.model_name}: {e}")
            raise
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            List of texts to embed
        Returns: 
            numpy array of embeddings with shape
        """

        if not self.model_name:
            raise ValueError("Model not loaded yet!")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        
        return embeddings

embedding_manager = EmbeddingManager()
embedding_manager

Loading Embedded model: all-MiniLM-L6-v2
Model loaded successfully. Embedding dimension: 384


### VectorStore 

In [9]:
class VectorStore:
    """
    Manages document embeddings in a ChromaDb vector store
    """
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store") -> None:
        """
        Initialize the Vector Store

        Args:
            collection_name: Name of the ChromaDb collection
            persist_directory: Directory to the persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
    
    def _initialize_store(self):
        """  
            Initialize the ChromaDb Client and collection
        """
        try:
            # Create persistant chromadb client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or Create a collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF embedding document for RAG"}
            )

            print(f"Vector store is initialized: Collection name: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error while initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[str], embeddings: np.ndarray) -> None:
        """  
        Add documents and its embeddings into vector store

        Args:
            documents: List[str] -> List of Chunks of the pdf documents
            embeddings: np.ndarrray -> Corresponding embeddings for the document chunks
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} into the vector store...")

        # prepare data to store
        id = []
        metadata = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embed) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            id.append(doc_id)

            # prepare metadata 
            pre_metadata = dict(doc.metadata)
            pre_metadata['doc_length'] = i
            pre_metadata['content_length'] = len(doc.page_content)
            metadata.append(pre_metadata)

            # document content
            documents_text.append(doc.page_content)

            # embedding content
            embeddings_list.append(embed.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=id,
                metadatas=metadata,
                documents=documents_text,
                embeddings=embeddings_list
            )
            print(f"Successfully added {len(documents)} documents to the vector store!")
            print(f"Total documents in vector store is: {self.collection.count()}")
        
        except Exception as e:
            print(f"Error while adding data into the vector store: {e}")
            raise

vector_store = VectorStore()
vector_store

Vector store is initialized: Collection name: pdf_documents
Existing documents in collection: 79


In [9]:
# Convert Text to embeddings
Texts = [doc.page_content for doc in chunks]

# Pass the Texts into the embeddings
embeddings = embedding_manager.generate_embeddings(texts=Texts)
print(type(embeddings))

Generating embeddings for 79 texts...


Batches: 100%|██████████| 3/3 [00:03<00:00,  1.02s/it]

Generated embeddings with shape: (79, 384)
<class 'numpy.ndarray'>


In [ ]:
print(embeddings['documents'])

[[ 0.00316992  0.04415252 -0.04119875 ... -0.0275773   0.04719916
   0.01611139]
 [-0.00144696  0.03045261 -0.00375371 ... -0.04045176  0.06935032
   0.00177604]
 [-0.01383999 -0.00502507 -0.05279252 ...  0.0228582   0.11088973
   0.0187209 ]
 ...
 [ 0.03158993 -0.09446768 -0.02935368 ... -0.05752939 -0.02952754
   0.13762285]
 [ 0.04287827 -0.00432632 -0.02394694 ... -0.0812654  -0.02752957
   0.04073236]
 [-0.06904999  0.0246567  -0.00385926 ... -0.01883772  0.01547315
   0.04743891]]


In [13]:
# Pass the text & its corresponding embeddings into the Vector store
vector_store.add_documents(documents=chunks, embeddings=embeddings)

Adding 79 into the vector store...


Successfully added 79 documents to the vector store!
Total documents in vector store is: 79


### Retriver Pipeline from vectorStore

In [12]:
class RAGRetriever:
    """  
        Handles query based retrieval from the vector store
    """
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """  
        Initialize the Retriever
        
        Args:
            vector_store: passing the vector store to access it
            embedding_manager: passing the EmbeddingManager object to access its methods
        """
        self.vector_store =  vector_store
        self.embedding_manager = embedding_manager
    
    def retriever(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """  
        Retrieve relevant document for the given query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for the query: {query}")
        print(f"K-Tops: {top_k}, Score threshold value: {score_threshold}")

        # generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings(texts=[query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=query_embedding,
                n_results=top_k
            )
            # print(results)

            # process results
            retreived_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # convert distance to similarity score (ChromaDb uses cosine similarity)
                    similarity_score = 1 - distance

                    if similarity_score > score_threshold:
                        retreived_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retreived_docs)} documents after filtering")
            else:
                print("No document found")

            return retreived_docs
        
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever = RAGRetriever(vector_store, embedding_manager)
 

In [21]:
rag_retriever.retriever(query="What is stock market")

Retrieving documents for the query: What is stock market
K-Tops: 5, Score threshold value: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 33.10it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents after filtering


[{'id': 'doc_1fbb94d3_71',
  'content': 'Stock Market System \n• Primary market \n• stock market is a secondary market \n• trade stock for listed corporations \n• trade stock for listed corporations \n• Progressive development of stock \nmarket',
  'metadata': {'producer': 'PDF Converter Elite',
   'source': '..\\data\\pdf\\Stock Market.pdf',
   'creator': 'PDF Converter Elite',
   'page': 7,
   'page_label': '8',
   'author': 'Ashish',
   'source_file': 'Stock Market.pdf',
   'subject': '',
   'creationdate': '2010-02-01T20:12:11+05:30',
   'file_type': 'pdf',
   'title': '',
   'doc_length': 71,
   'moddate': '2010-02-01T20:12:11+05:30',
   'content_length': 197,
   'total_pages': 15,
   'keywords': ''},
  'similarity_score': 0.21929192543029785,
  'distance': 0.7807080745697021,
  'rank': 1},
 {'id': 'doc_b56f3ebd_69',
  'content': 'Why Trade In Stock Market \n• 1. You do not need a lot of money to start making money, unlike buying \nproperty and paying a monthly mortgage. \n• 2. It

### Integration vectorDB context pipeline with LLM

In [2]:
## Checking DeepSeek model with OpenAI package
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv()

api_key = os.getenv("GENAI_API_KEY")

# Check whether the api_key got value
if not api_key:
    print("[INFO] API key not found in environment variables")
    raise
else:
    print("[INFO] API key is successfully loaded!")

# Initialize the ChatOpenAI model with OpenRouter's details
llm_model = ChatOpenAI(
    model="deepseek/deepseek-r1-0528:free",  # The specific model name from OpenRouter
    base_url="https://openrouter.ai/api/v1", # OpenRouter's API endpoint
    api_key=api_key
)

messages = [
    (
        "system",
        "You are a helpful assistant that translates English to Tamil. Translate the user sentence.",
    ),
    ("human", "Hey, Give few words about stock market"),
]

response = llm_model.invoke(messages)

print(response)

[INFO] API key is successfully loaded!


ValueError: {'message': 'Network connection lost.', 'code': 502, 'metadata': {'provider_name': 'ModelRun'}}

In [4]:
print(api_key)

None


In [16]:
## Create a simple RAG function to retrieve context and get response from LLM
def rag_simple(query: str, retriever: RAGRetriever = rag_retriever, k_top: int = 3, llm_model=llm_model) -> str: 
    """  
        Get the query from the user, retrieve required documents from vector store and given to
        the llm-model to get response which would printed 

        Args:
            query -> User Query,
            retreiver -> RAGRetriever object
            k_top -> tells how many similar documents need to be retrieve from vector store
            llm_model -> llm model setup
    """
    results = rag_retriever.retriever(query, k_top)
    context = "\n\n".join([doc["content"] for doc in results]) if results else ""

    if not context:
        return "No relevent context is retrieved to answer the user query"
    
    prompt = f"""Use the following context to answer the given question precisely
            context : 
            {context}

            question:
            {query}

            Answer:"""

    response = llm_model.invoke([prompt.format(context=context, query=query)])
    return response.content

print(rag_simple("What is stock market?"))

    


Retrieving documents for the query: What is stock market?
K-Tops: 3, Score threshold value: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 32.87it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents after filtering


Based strictly on the provided context, the stock market is defined as:

1.  **A Secondary Market:** It's specifically identified as a secondary market.
2.  **Where Trading Occurs:** Its primary circled function is to **trade stocks for listed corporations**.
3.  **Facilitated by Exchanges/Regulators:** It consists of **Exchanges, Indices, [SEBI]** (the regulatory body mentioned).

**Therefore, the precise answer from the context is:**

**The stock market is a secondary market where stocks of listed corporations are traded, consisting of exchanges, indices, and regulators like SEBI.**

### Key Details Confirmed by the Context
*   **Secondary Market:** Explicitly stated ("stock market is a secondary market").
*   **Trading Listed Stocks:** Repeatedly emphasized ("trade stock for listed corporations").
*   **Components:** The question "What does the Share Market consist of?" is answered with "Exchanges, Indices, SEBI".
*   **Excludes:** The context distinguishes the stock market (seconda

In [ ]:
import os
import shutil

def move_all_files(source_folder: str, destination_folder: str) -> None:
    """
    Copies all files from source (including subfolders)
    to destination and deletes the original files.
    Args:
        source_folder: str
        destination_folder: str
    """

    for root, dirs, files in os.walk(source_folder):
        for file in files:
            source_file_path = os.path.join(root, file)

            # Preserve folder structure
            relative_path = os.path.relpath(root, source_folder)
            destination_dir = os.path.join(destination_folder, relative_path)

            os.makedirs(destination_dir, exist_ok=True)

            destination_file_path = os.path.join(destination_dir, file)

            try:
                # Copy file
                shutil.copy2(source_file_path, destination_file_path)

                # Delete original file after successful copy
                os.remove(source_file_path)

                print(f"[INFO] Moved: {source_file_path}")

            except Exception as e:
                print(f"[ERROR] failed while moving {source_file_path}: {e}")

    print("[INFO] Loaded files are moved successfully!")


# Example usage
source = r"..\data"
destination = r"..\archive"

move_all_files(source, destination)

Moved: ..\data\csv\Sold Records.csv
Moved: ..\data\pdf\Money Market.pdf
Moved: ..\data\pdf\Stock Market.pdf
Moved: ..\data\txt\ml_intro.txt
Moved: ..\data\txt\python_intro.txt
Moved: ..\data\vector_store\chroma.sqlite3
Moved: ..\data\vector_store\8b8a3e93-7c74-418b-954d-a8b42de225bd\data_level0.bin
Moved: ..\data\vector_store\8b8a3e93-7c74-418b-954d-a8b42de225bd\header.bin
Moved: ..\data\vector_store\8b8a3e93-7c74-418b-954d-a8b42de225bd\length.bin
Moved: ..\data\vector_store\8b8a3e93-7c74-418b-954d-a8b42de225bd\link_lists.bin
All files moved successfully!


In [9]:
import os
import shutil

source_dir = '../data'  # Replace with your source path
destination_dir = '../archive' # Replace with your destination path

# Create destination directory if it doesn't exist
if not os.path.exists(destination_dir):
    os.makedirs(destination_dir, exist_ok=True)

# Move all contents
files_to_move = os.listdir(source_dir)
print(files_to_move)
for file_name in files_to_move:
    source_path = os.path.join(source_dir, file_name)
    
    try:
        des_path = os.path.join(destination_dir, file_name)
        if not os.path.exists(des_path):
            shutil.move(source_path, des_path)
            print(f"Moved: {file_name}")
        else:
            with open(source_path, 'r') as src:
                # Open destination file in append mode ('a')
                with open(des_path, 'a') as dst:
                    # Efficiently copy contents
                    shutil.copyfileobj(src, dst)

        print(f"Content of {source_path} appended to {destination_dir}")
    except shutil.Error as e:
        print(f"Error moving {file_name}: {e}")
    except OSError as e:
        print(f"OS error moving {file_name}: {e}")

print("All items moved successfully.")

['txt']
OS error moving txt: [Errno 13] Permission denied: '../data\\txt'
All items moved successfully.
